---
layout: post
codemirror: true
title: Easy API - Jokes HW
categories: [Flask, Python, APIs]
permalink: /python/flask/jokes-api-hw
author: rex0dex
---

## Easy API - Jokes

Aaditya Prem (rex0dex). The point of this lesson is to learn how to **read** a backend, not just call it: start with a request, find the handler, follow the data call, and check that the response matches. I did it on our team's Flask repo (`usc-flask`), running locally on port **8588**.

| Task | What I did |
| --- | --- |
| Start with a request | Sent the two public GETs (`/count` and the list) and pasted the real responses |
| Read `hacks/joke.py` | Matched every `add_resource` line to its class, method, full URL, and data function |
| Follow `hacks/jokes.py` | Traced request -> handler -> data function -> `jokes.json` -> response for both reads |
| Object vs array | Explained why `/count` is `{}` and the list is `[]`, and checked it in Python |
| Other operations | Read random, by id, like, jeer, and the unfinished create, plus tested bad ids with GET only |
| Reading Routine | Applied steps 1-5 from the Flask APIs overview to this API |

I only sent **GET** requests. Like and jeer change stored votes, so I read their code instead of sending them.

## Start with a Request

The repo's `.vscode/settings.json` gives REST Client a `baseUrl` of `http://127.0.0.1:8587` (in `$shared`), and the `local` environment is empty, so it inherits that. My backend runs on **8588**, so `baseUrl` has to match 8588 or REST Client just says connection refused. The `.http` file has no `@baseUrl` line, so it doesn't override the environment.

{% raw %}

```http
### Count jokes - expect 200 and an object with count
GET {{baseUrl}}/api/jokes/count
Accept: application/json

### Read jokes - expect 200 and an array
GET {{baseUrl}}/api/jokes
Accept: application/json
```

{% endraw %}

These two blocks are also the first two requests in `hacks/joke.http`, which Mr. Mortensen added to the repo.

**Real response, `GET /api/jokes/count`:**

    HTTP/1.1 200 OK
    Server: Werkzeug/3.1.9 Python/3.12.10
    Content-Type: application/json
    Content-Length: 18

    {
      "count": 18
    }

**Real response, `GET /api/jokes` (first 2 of 18 items):**

    HTTP/1.1 200 OK
    Server: Werkzeug/3.1.9 Python/3.12.10
    Content-Type: application/json
    Content-Length: 2847

    [
      {
        "boohoo": 0,
        "haha": 0,
        "id": 0,
        "joke": "If you give someone a program... you will frustrate them for a day; if you teach them how to program... you will frustrate them for a lifetime."
      },
      {
        "boohoo": 1,
        "haha": 0,
        "id": 1,
        "joke": "Q: Why did I divide sin by tan? A: Just cos."
      },
      ...
    ]

What I checked (not just "it connected"):

* **Status:** both are `200 OK`.
* **Headers:** both are `Content-Type: application/json`, so the body is JSON, not an HTML page.
* **Body:** `/count` is an **object** with one key, `count`. The list is an **array** of 18 objects, each with `id`, `joke`, `haha`, and `boohoo`.
* No login or cookie was needed. These are public reads.

## Read hacks/joke.py

**The blueprint (lines 8-9):** `joke_api = Blueprint('joke_api', __name__, url_prefix='/api/jokes')`. Every route in this file starts with `/api/jokes`. Line 12 wraps it in a Flask-RESTful `Api(joke_api)`, and `main.py` line 118 registers it on the app with `app.register_blueprint(joke_api)`.

**The resources (lines 55-61):** full URL = prefix + resource path. Each one is registered twice, with and without a trailing `/`.

| Line | Resource path | Class | Method | Full URL | Calls in `jokes.py` |
| --- | --- | --- | --- | --- | --- |
| 55 | `/create/<string:joke>` | `_Create` | `post` | `/api/jokes/create/<joke>` | nothing (`pass`) |
| 56 | `""` and `/` | `_Read` | `get` | `/api/jokes` | `getJokes()` |
| 57 | `/<int:id>` | `_ReadID` | `get` | `/api/jokes/<id>` | `getJoke(id)` |
| 58 | `/random` | `_ReadRandom` | `get` | `/api/jokes/random` | `getRandomJoke()` |
| 59 | `/count` | `_ReadCount` | `get` | `/api/jokes/count` | `countJokes()` |
| 60 | `/like/<int:id>` | `_UpdateLike` | `put` | `/api/jokes/like/<id>` | `addJokeHaHa(id)`, then `getJoke(id)` |
| 61 | `/jeer/<int:id>` | `_UpdateJeer` | `put` | `/api/jokes/jeer/<id>` | `addJokeBooHoo(id)`, then `getJoke(id)` |

The method name inside the class (`get`, `put`, `post`) is what decides the HTTP verb. So `/api/jokes/count` is handled by **`_ReadCount.get`** (lines 36-40) and `/api/jokes` is handled by **`_Read.get`** (lines 21-23).

### Why /count is an object and the list is an array

```python
class _Read(Resource):          # lines 21-23
    def get(self):
        return jsonify(getJokes())

class _ReadCount(Resource):     # lines 36-40
    def get(self):
        count = countJokes()
        countMsg = {'count': count}
        return jsonify(countMsg)
```

`jsonify` just turns whatever Python value it gets into JSON. The shape of the response is decided by what the handler passes in:

* `countJokes()` returns an **int** (18). A bare number isn't very descriptive, so `_ReadCount.get` wraps it in a dictionary `{'count': count}`. A Python **dict** becomes a JSON **object**: `{"count": 18}`.
* `getJokes()` returns the whole **list** of joke dictionaries, and `_Read.get` passes it straight through. A Python **list** becomes a JSON **array**: `[{...}, {...}, ...]`.

So the difference isn't Flask doing something different. One handler builds a dict and the other hands over a list.

## Follow hacks/jokes.py

`jokes.py` is the data layer for this example. There is no SQLAlchemy model or table, just a JSON file plus functions that read and write it.

**Where the file is:**

* `get_jokes_file()` (lines 28-31) reads `current_app.config['DATA_FOLDER']` and joins it with `'jokes.json'`. It uses Flask's `current_app`, so it only works while a request (or app context) is active.
* `DATA_FOLDER` is set in `__init__.py` line 140: `os.path.join(app.instance_path, 'data')`. On my machine that is `usc-flask/instance/data/jokes.json`.

**The read helper:** `_read_jokes_file()` (lines 33-44) returns `[]` if the file doesn't exist, otherwise opens it, takes a **shared lock** (`fcntl.LOCK_SH`) so many readers can read at once, runs `json.load(f)`, and unlocks. If the JSON is broken it also falls back to `[]`.

**The two functions I traced:**

* `getJokes()` (lines 72-73) is just `return _read_jokes_file()`, so it returns the list.
* `countJokes()` (lines 128-130) reads the same list and returns `len(jokes)`, an int.

**Where the data comes from:** `initJokes()` (lines 53-70) is called once at startup in `main.py` lines 128-129 inside `with app.app_context():`. It returns early if `jokes.json` already exists. Otherwise it builds one dictionary per string in `joke_list` (lines 5-26, 18 jokes) with `id`, `joke`, `haha: 0`, `boohoo: 0`, then randomly adds **10** haha votes and **5** boohoo votes, and writes it with `_write_jokes_file()` (lines 46-51, exclusive lock).

### My trace

    GET /api/jokes/count
      -> joke_api blueprint, prefix /api/jokes          (joke.py 8-9)
      -> _ReadCount.get                                 (joke.py 36-40, registered 59)
      -> countJokes()                                   (jokes.py 128-130)
      -> _read_jokes_file() -> get_jokes_file()         (jokes.py 33-44, 28-31)
      -> instance/data/jokes.json  (list of 18)
      -> len(...) = 18 -> {'count': 18} -> jsonify
      -> 200, application/json, {"count": 18}

    GET /api/jokes
      -> joke_api blueprint, prefix /api/jokes          (joke.py 8-9)
      -> _Read.get                                      (joke.py 21-23, registered 56)
      -> getJokes()                                     (jokes.py 72-73)
      -> _read_jokes_file() -> get_jokes_file()         (jokes.py 33-44, 28-31)
      -> instance/data/jokes.json  (list of 18)
      -> jsonify(list)
      -> 200, application/json, [ {...}, ... 18 items ]

The cell below checks the trace end to end: the response types, that both reads agree, and that the list is exactly what is in `jokes.json`.

In [1]:
# GET requests only against my local backend (no like/jeer/create)
import json, os
from urllib.request import urlopen, Request

BASE = "http://localhost:8588/api/jokes"

def get(path):
    """GET BASE + path; return (status, content type, parsed JSON body)."""
    req = Request(BASE + path, headers={"Accept": "application/json"})
    with urlopen(req) as resp:
        return resp.status, resp.headers["Content-Type"], json.loads(resp.read())

# /count -> _ReadCount.get -> countJokes()
status, ctype, count_body = get("/count")
print("GET /api/jokes/count ->", status, ctype)
print("   body:", count_body, "-> Python", type(count_body).__name__)

# "" -> _Read.get -> getJokes()
status, ctype, jokes = get("")
print("GET /api/jokes       ->", status, ctype)
print("   Python", type(jokes).__name__, "of", len(jokes), "items, keys:", sorted(jokes[0]))

# /<int:id> and /random also return ONE joke (an object)
status, _, one = get("/7")
print("GET /api/jokes/7     ->", status, "-> Python", type(one).__name__, "id", one["id"])
status, _, rand = get("/random")
print("GET /api/jokes/random ->", status, "-> Python", type(rand).__name__, "id", rand["id"])

# The two reads agree, and the list is exactly what is in the JSON file
print("\ncount == len(list)?", count_body["count"] == len(jokes))
path = os.path.expanduser("~/rex0dex/usc-flask/instance/data/jokes.json")
with open(path) as f:
    on_disk = json.load(f)
print("API list == instance/data/jokes.json?", jokes == on_disk)
print("key order in the file:", list(on_disk[0]), "| in the response:", list(jokes[0]))

# Votes so far
top = max(j["haha"] for j in jokes)
print("\ntotal haha:", sum(j["haha"] for j in jokes), "| total boohoo:", sum(j["boohoo"] for j in jokes))
print("most haha:", top, "-> ids", [j["id"] for j in jokes if j["haha"] == top])

GET /api/jokes/count -> 200 application/json
   body: {'count': 18} -> Python dict
GET /api/jokes       -> 200 application/json
   Python list of 18 items, keys: ['boohoo', 'haha', 'id', 'joke']
GET /api/jokes/7     -> 200 -> Python dict id 7
GET /api/jokes/random -> 200 -> Python dict id 16

count == len(list)? True
API list == instance/data/jokes.json? True
key order in the file: ['id', 'joke', 'haha', 'boohoo'] | in the response: ['boohoo', 'haha', 'id', 'joke']

total haha: 10 | total boohoo: 5
most haha: 2 -> ids [2, 17]


What the output shows:

* `/count` parses to a Python `dict` and the list parses to a Python `list`, which matches the object/array explanation. `/7` and `/random` are single jokes, so they come back as `dict` too.
* `count == len(list)` is `True`, because `countJokes()` is literally `len()` of the same list `getJokes()` returns.
* The API list equals what `json.load` reads from `instance/data/jokes.json`, so the response really is the file. The only difference is key order: the file has `id, joke, haha, boohoo` (the order `initJokes` built them), but the response has them alphabetical because `jsonify` sorts keys.
* Totals are exactly **10 haha** and **5 boohoo**, the same numbers `initJokes()` primes. That tells me nobody has voted on my local copy yet, which makes sense since I never sent like or jeer. Two jokes are tied for most haha (ids 2 and 17).

## Read Before Trying Other Operations

**`_ReadRandom.get` (lines 31-33):** `getRandomJoke()` (jokes.py 79-81) does `random.choice(jokes)`, so it returns one object and a different one each time.

**`_ReadID.get` (lines 26-28):** `getJoke(id)` (jokes.py 75-77) is just `return jokes[id]`. There is no check that `id` is in range.

**`_UpdateLike.put` / `_UpdateJeer.put` (lines 43-52):** these are `PUT`, and they **change stored data**. `addJokeHaHa(id)` and `addJokeBooHoo(id)` (lines 119-123) both call `_vote_joke(id, field)` (lines 105-117), which opens the file with `'r+'`, takes an **exclusive lock**, adds 1 to `jokes[id][field]`, then `seek(0)`, `json.dump`, and `truncate()` so no old bytes are left at the end. Then the handler calls `getJoke(id)` again and returns that one joke as an object. I didn't send these.

**`_Create.post` (lines 16-18):** the comment says "not implemented" and the body is `pass`, so it returns nothing and nothing is saved. Also, `jokes.py` has no function that adds a new joke at all (the only writes are `initJokes` and `_vote_joke`), and the joke text would come from the URL path (`/create/<string:joke>`) instead of a JSON body. The route is registered, but registered does not mean finished. This API is really Read plus a vote Update, not full CRUD (no real Create, no Delete).

**`hacks/joke.http`:** besides the two reads, it has "Read one joke" (`/api/jokes/14`) and like/jeer `PUT`s on id 0. Their comments say "expect 200 and an array", but from the code these return **one joke object**, not an array. Also, the `if __name__ == "__main__"` block in `joke.py` (lines 63-97) sends `PUT` like and jeer to the deployed server, so running that file directly would vote on the real site.

To check the "no friendly response for every invalid ID" part without changing anything, I sent GETs to some bad paths:

In [2]:
# Edge cases on the read routes (still GET only)
from urllib.request import urlopen
from urllib.error import HTTPError

BASE = "http://localhost:8588/api/jokes"

for path in ["/17", "/18", "/abc", "/-1", "/create/hi"]:
    try:
        with urlopen(BASE + path) as resp:
            print(f"GET {path:<11} -> {resp.status} {resp.headers['Content-Type']}")
    except HTTPError as err:
        print(f"GET {path:<11} -> {err.code} {err.headers['Content-Type']}")

GET /17         -> 200 application/json
GET /18         -> 500 text/html; charset=utf-8
GET /abc        -> 404 text/html; charset=utf-8
GET /-1         -> 404 text/html; charset=utf-8
GET /create/hi  -> 405 application/json


| Request | Status | Why |
| --- | --- | --- |
| `/17` | 200 JSON | Last valid id (ids are 0-17) |
| `/18` | **500**, HTML | Matches `<int:id>`, so `_ReadID.get` runs, but `jokes[18]` raises `IndexError` at jokes.py line 77. Nothing catches it, so the server crashes the request and sends an HTML error page instead of JSON |
| `/abc` | 404 | `<int:id>` doesn't match letters, so no handler runs |
| `/-1` | 404 | The `int` converter doesn't accept negatives, so `jokes[-1]` (which in Python would quietly return the *last* joke) is never reached |
| `/create/hi` | 405, JSON | The route exists, but `_Create` only defines `post`, so a GET is "method not allowed" |

So the only bad id that reaches the code is one that is too big, and it gets a 500 instead of a friendly 404 with a message. A real fix would be checking `0 <= id < len(jokes)` in `getJoke` and returning `{'message': 'Joke not found'}, 404`. I'm not changing the code for this homework, just noting it.

## Reading Routine (from the Flask APIs overview)

| Step | Applied to the Jokes API |
| --- | --- |
| 1. Blueprint prefix and resource registration | `url_prefix='/api/jokes'` (joke.py 8-9), resources registered on lines 55-61, blueprint registered in main.py line 118 |
| 2. Method handling the request, auth or permissions | `_ReadCount.get` and `_Read.get` for my two reads. No `@token_required` or any auth import in `joke.py`, so every route is public, including the `PUT`s that change votes |
| 3. Input fields, query parameters, validation failures | No JSON body and no query parameters. The only inputs are path parameters: `<int:id>` (like, jeer, by id) and `<string:joke>` (create). Validation is just the URL converter: non-int or negative is 404, too-big id is a 500 `IndexError` |
| 4. Follow the data call to its input and return value | `countJokes()` takes nothing and returns an int; `getJokes()` takes nothing and returns a list of dicts. Both go through `_read_jokes_file()` to `DATA_FOLDER/jokes.json` |
| 5. Compare status and response structure with REST Client output | Both reads are `200 application/json`; `/count` is `{"count": 18}` (dict wrapped by the handler) and the list is an 18-item array, which matches the code |

## What I Learned

* The URL is two pieces in two places: the blueprint prefix plus the `add_resource` path. I need both to know which class handles a request.
* The response shape comes from the handler, not the data function. `countJokes()` returns an int, and the handler is what makes it an object.
* A "data layer" doesn't have to be a database. Here it is a JSON file with file locks, shared for reading and exclusive for writing, so two votes at the same time don't overwrite each other.
* A registered route isn't proof of a working feature (`_Create` is `pass`), and an API doesn't always validate input (`/18` is a 500). Read the handler before sending anything that writes.
* Next is the User API, which adds authentication and a real SQLAlchemy model behind the handler.